In [ ]:
import requests
import urllib.parse
import time
from typing import Dict, Any
class DanishParliamentAPI:
    def __init__(self, timeout: int = 30, retry_attempts: int = 3):
        """
        Initialize the API client.

        Args:
            timeout: Request timeout in seconds
            retry_attempts: Number of retry attempts for failed requests
        """
        self.base_url = "https://oda.ft.dk/api/"
        self.timeout = timeout
        self.retry_attempts = retry_attempts
        self.last_request_time = 0
        self.min_request_interval = 0.1  # Minimum 100ms between requests

    def _rate_limit(self) -> None:
        """Enforce rate limiting between requests."""
        elapsed = time.time() - self.last_request_time
        if elapsed < self.min_request_interval:
            time.sleep(self.min_request_interval - elapsed)
        self.last_request_time = time.time()

    def _make_request(self, url: str) -> Dict[str, Any]:
        """
        Make HTTP request with retry logic and error handling.

        Args:
            url: Complete URL to request

        Returns:
            Parsed JSON response

        Raises:
            APIError: For various API errors
            NetworkError: For network-related errors
        """
        self._rate_limit()

        for attempt in range(self.retry_attempts):
            try:
                response = requests.get(url, timeout=self.timeout)

                # Handle different HTTP status codes
                if response.status_code == 200:
                    return response.json()
                elif response.status_code == 400:
                    raise APIError(
                        f"Invalid query parameters. Check $expand and $filter syntax. "
                        f"URL: {url}"
                    )
                elif response.status_code == 404:
                    if 'api/' in url and url.count('/') == 4:  # Entity not found
                        raise EntityNotFoundError(f"Entity not found: {url}")
                    else:  # Invalid ID
                        raise RecordNotFoundError(f"Record not found: {url}")
                elif response.status_code == 501:
                    raise UnsupportedOperationError(
                        "Write operations are not supported by this API"
                    )
                else:
                    response.raise_for_status()

            except requests.exceptions.Timeout:
                if attempt < self.retry_attempts - 1:
                    wait_time = (2 ** attempt) * 1  # Exponential backoff
                    time.sleep(wait_time)
                    continue
                raise NetworkError(f"Request timed out after {self.timeout} seconds")

            except requests.exceptions.ConnectionError:
                if attempt < self.retry_attempts - 1:
                    wait_time = (2 ** attempt) * 1
                    time.sleep(wait_time)
                    continue
                raise NetworkError("Connection error - check your internet connection")

            except requests.exceptions.RequestException as e:
                raise NetworkError(f"Request failed: {str(e)}")

    def _build_url(self, entity: str, **params) -> str:
        """
        Build properly encoded URL with OData parameters.

        Args:
            entity: Entity name (e.g., 'Sag', 'Aktør')
            **params: OData parameters

        Returns:
            Complete URL with encoded parameters
        """
        # Start with base URL and entity
        url = f"{self.base_url}{entity}"

        if not params:
            return url

        # Build query parameters with proper encoding
        query_parts = []
        for key, value in params.items():
            if value is not None:
                # Ensure $ parameters are properly encoded
                if key.startswith('$'):
                    encoded_key = urllib.parse.quote(key, safe='$')
                else:
                    encoded_key = key

                encoded_value = urllib.parse.quote(str(value), safe="(),'/%")
                query_parts.append(f"{encoded_key}={encoded_value}")

        return f"{url}?{'&'.join(query_parts)}"



# Custom Exception Classes
class APIError(Exception):
    """Base exception for API errors."""
    pass

class NetworkError(APIError):
    """Network-related errors."""
    pass

class EntityNotFoundError(APIError):
    """Entity does not exist."""
    pass

class RecordNotFoundError(APIError):
    """Specific record does not exist."""
    pass

class UnsupportedOperationError(APIError):
    """Operation not supported by API."""
    pass

Start by fetching all the voting procedure. Each row represents one voting procedure in the Folketing, in the API the 'Afstemning'-tag only gets very limited info, but we expand the Møde and Sagstrin columns to get info about the session the vote took place in as well as the case-phase which tells us about the proposal being voted for.

In [ ]:
api = DanishParliamentAPI()

all_voting_data = []
skip = 0
batch_size = 100

while True:
    votings_response = api._make_request(api._build_url('Afstemning', **{
        '$top': batch_size, 
        '$expand': 'Møde,Sagstrin',
        '$skip': skip
    }))
    
    voting_data = votings_response.get('value', [])
    if not voting_data:
        break
    
    all_voting_data.extend(voting_data)
    skip += batch_size
    
    if skip % 500 == 0:
        print(f"  Fetched {skip} votings...")

print(f"Total votings fetched: {len(all_voting_data)}")
all_voting_data_df = pd.DataFrame(all_voting_data)


  Fetched 500 votings...
  Fetched 1000 votings...
  Fetched 1500 votings...
  Fetched 2000 votings...
  Fetched 2500 votings...
  Fetched 3000 votings...
  Fetched 3500 votings...
  Fetched 4000 votings...
  Fetched 4500 votings...
  Fetched 5000 votings...
  Fetched 5500 votings...
  Fetched 6000 votings...
  Fetched 6500 votings...
  Fetched 7000 votings...
  Fetched 7500 votings...
  Fetched 8000 votings...
  Fetched 8500 votings...
  Fetched 9000 votings...
  Fetched 9500 votings...
  Fetched 10000 votings...
Total votings fetched: 10306


The Field from Møde and Sagstrin are nested by default, so we flatten those columns being careful to label the columns which are named similarly across the three different databases.

In [9]:
def flatten_voting(v: Dict[str, Any]) -> Dict[str, Any]:
    """Flatten a single voting record with all nested data."""
    flat = {
        # Afstemning fields
        'afstemning_id': v.get('id'),
        'afstemning_nummer': v.get('nummer'),
        'afstemning_konklusion': v.get('konklusion'),
        'afstemning_vedtaget': v.get('vedtaget'),
        'afstemning_kommentar': v.get('kommentar'),
        'afstemning_dato': v.get('dato'),
        'afstemning_opdateringsdato': v.get('opdateringsdato'),
        'afstemning_typeid': v.get('typeid'),
        
        # From Møde (nested)
        'møde_id': None,
        'møde_dato': None,
        'møde_titel': None,
        'møde_nummer': None,
        'møde_periodeid': None,
        
        # From Sagstrin (nested)
        'sagstrin_id': None,
        'sagstrin_titel': None,
        'sagstrin_typeid': None,
        'sag_id': None,
    }
    
    # Extract Møde fields
    møde = v.get('Møde')
    if isinstance(møde, dict):
        flat['møde_id'] = møde.get('id')
        flat['møde_dato'] = møde.get('dato')
        flat['møde_titel'] = møde.get('titel')
        flat['møde_nummer'] = møde.get('nummer')
        flat['møde_periodeid'] = møde.get('periodeid')
    
    # Extract Sagstrin fields
    sagstrin = v.get('Sagstrin')
    if isinstance(sagstrin, dict):
        flat['sagstrin_id'] = sagstrin.get('id')
        flat['sagstrin_titel'] = sagstrin.get('titel')
        flat['sagstrin_typeid'] = sagstrin.get('typeid')
        flat['sag_id'] = sagstrin.get('sagid')
    
    return flat

# Flatten all votings
flattened_votings = [flatten_voting(v) for v in all_voting_data]

In [12]:
import pandas as pd
votings_df = pd.DataFrame(flattened_votings)

In [ ]:
votings_df[votings_df['afstemning_nummer'] == 464]

,afstemning_id,afstemning_nummer,afstemning_konklusion,afstemning_vedtaget,afstemning_kommentar,afstemning_dato,afstemning_opdateringsdato,afstemning_typeid,møde_id,møde_dato,møde_titel,møde_nummer,møde_periodeid,sagstrin_id,sagstrin_titel,sagstrin_typeid,sag_id
659,662,464,"Vedtaget\n\n92 stemmer for forslaget (V, S, RV...",True,,None,2018-01-25T10:12:08.6,1,1520,2013-06-04T09:00:00,Møde i salen,108,31,9791.0,3. behandling,17.0,3102.0
1770,1777,464,"\nForkastet\n\n50 stemmer for forslaget (S, SF...",False,,None,2018-01-25T16:57:03.783,4,3451,2011-06-21T10:00:00,Møde i salen,106,28,47305.0,2. behandling,15.0,15878.0
2214,2233,464,,True,,None,2018-02-09T16:08:18.413,1,2624,2010-06-04T09:00:00,Møde i Salen,104,27,57371.0,3. behandling,17.0,18629.0
3063,3106,464,"Forkastet\n\n47 stemmer for forslaget (DF, EL,...",False,None,None,2016-05-31T13:54:01.477,1,5081,2016-05-31T13:00:00,Møde i salen,104,139,171176.0,2. (sidste) behandling,7.0,68783.0
3640,3693,464,"Vedtaget\n\n94 stemmer for forslaget (S, DF, V...",True,None,None,2017-06-02T09:40:00.26,1,7212,2017-06-02T09:00:00,Møde i salen,107,144,180703.0,3. behandling,17.0,72429.0
4308,4365,464,,True,,None,2018-02-12T14:30:28.967,1,6089,2009-05-29T09:00:00,Møde i Salen,101,26,152657.0,3. behandling,17.0,21011.0
5129,5186,464,,True,,None,2018-02-15T12:01:54.227,1,5772,2007-05-30T13:00:00,Møde i Salen,98,23,122324.0,3. behandling,17.0,47661.0
5568,5625,464,,True,,None,2018-02-15T17:20:53.1,1,5602,2006-06-02T09:00:00,Møde i salen,98,22,96906.0,3. behandling,17.0,37458.0
6595,6658,464,"Vedtaget\n\n53 stemmer for forslaget (DF, V, L...",True,None,None,2018-06-01T10:26:18.557,1,8153,2018-06-01T10:00:00,Møde i salen,106,146,192985.0,3. behandling,17.0,76284.0
7951,8028,464,"Forslaget blev forkastet. For stemte 37 (V, DF...",False,None,None,2021-04-20T08:27:37.273,1,10367,2020-06-25T09:20:00,Møde i salen,138,151,205478.0,2. (sidste) behandling,7.0,80877.0


In [ ]:
votings_df

Every sagstrin (case-phase) is part of a sag (case) which contains further info including the title and resume of the case, we find all the sag_ids that are referenced in our df and merge that info back into our dataframe.

In [ ]:
unique_sagids = votings_df['sag_id'].dropna().unique()
print(f"\nUnique cases to fetch: {len(unique_sagids)}")

all_sag_data = []
for i, sagid in enumerate(unique_sagids):
    if i % 100 == 0:
        print(f"  Fetched {i}/{len(unique_sagids)} cases...")
    
    try:
        sag_response = api._make_request(api._build_url('Sag', **{
            '$filter': f'id eq {int(sagid)}'
        }))
        
        sag_data = sag_response.get('value', [])
        if sag_data:
            all_sag_data.extend(sag_data)
    except Exception as e:
        print(f"  Error fetching Sag {sagid}: {e}")
        continue




Unique cases to fetch: 6871
  Fetched 0/6871 cases...
  Fetched 100/6871 cases...
  Fetched 200/6871 cases...
  Fetched 300/6871 cases...
  Fetched 400/6871 cases...
  Fetched 500/6871 cases...
  Fetched 600/6871 cases...
  Fetched 700/6871 cases...
  Fetched 800/6871 cases...
  Fetched 900/6871 cases...
  Fetched 1000/6871 cases...
  Fetched 1100/6871 cases...
  Fetched 1200/6871 cases...
  Fetched 1300/6871 cases...
  Fetched 1400/6871 cases...
  Fetched 1500/6871 cases...
  Fetched 1600/6871 cases...
  Fetched 1700/6871 cases...
  Fetched 1800/6871 cases...
  Fetched 1900/6871 cases...
  Fetched 2000/6871 cases...
  Fetched 2100/6871 cases...
  Fetched 2200/6871 cases...
  Fetched 2300/6871 cases...
  Fetched 2400/6871 cases...
  Fetched 2500/6871 cases...
  Fetched 2600/6871 cases...
  Fetched 2700/6871 cases...
  Fetched 2800/6871 cases...
  Fetched 2900/6871 cases...
  Fetched 3000/6871 cases...
  Fetched 3100/6871 cases...
  Fetched 3200/6871 cases...
  Fetched 3300/6871 cases.

In [ ]:
def flatten_sag(s: Dict[str, Any]) -> Dict[str, Any]:
    """Flatten a single Sag record."""
    return {
        'sag_id': s.get('id'),
        'sag_titel': s.get('titel'),
        'sag_titelkort': s.get('titelkort'),
        'sag_resume': s.get('resume'),
        'sag_nummer': s.get('nummer'),
        'sag_nummerprefix': s.get('nummerprefix'),
        'sag_opdateringsdato': s.get('opdateringsdato'),
        'sag_typeid': s.get('typeid'),
        'sag_periodeid': s.get('periodeid'),
        'sag_statusid': s.get('statusid'),
    }

flattened_sag = [flatten_sag(s) for s in all_sag_data]
sag_df = pd.DataFrame(flattened_sag)

In [27]:
if sag_df['sag_id'].duplicated().any():
    print(f"WARNING: {sag_df['sag_id'].duplicated().sum()} duplicate sag_ids found!")
    sag_df = sag_df.drop_duplicates(subset='sag_id', keep='first')

print(f"Sag dataframe shape: {sag_df.shape}")
enriched_votings = votings_df.merge(
    sag_df,
    on='sag_id',
    how='left'
)

print(f"\nFinal enriched shape: {enriched_votings.shape}")
print(f"Final columns: {enriched_votings.columns.tolist()}")

Sag dataframe shape: (6871, 10)

Final enriched shape: (10306, 26)
Final columns: ['afstemning_id', 'afstemning_nummer', 'afstemning_konklusion', 'afstemning_vedtaget', 'afstemning_kommentar', 'afstemning_dato', 'afstemning_opdateringsdato', 'afstemning_typeid', 'møde_id', 'møde_dato', 'møde_titel', 'møde_nummer', 'møde_periodeid', 'sagstrin_id', 'sagstrin_titel', 'sagstrin_typeid', 'sag_id', 'sag_titel', 'sag_titelkort', 'sag_resume', 'sag_nummer', 'sag_nummerprefix', 'sag_opdateringsdato', 'sag_typeid', 'sag_periodeid', 'sag_statusid']


In order to categorize the votes according to the parliamentary periods we need to massage the dates a little and compare them against a table of the periods.

In [ ]:
enriched_votings

In [ ]:
from folketingsperioder import periods_df
votings_df['møde_dato'] = pd.to_datetime(votings_df['møde_dato'])

# Create a mapping of Danish months to English
danish_months = {
    'januar': 'January', 'februar': 'February', 'marts': 'March',
    'april': 'April', 'maj': 'May', 'juni': 'June',
    'juli': 'July', 'august': 'August', 'september': 'September',
    'oktober': 'October', 'november': 'November', 'december': 'December'
}

def parse_date_range(date_range):
    dates = date_range.split(' – ')
    return dates[0].strip(), dates[1].strip()

def convert_danish_to_english(date_str):
    """Convert Danish month names to English."""
    for danish, english in danish_months.items():
        date_str = date_str.replace(danish, english)
    return date_str

# Create a mapping of periods with their date ranges
periods_df['start_date'] = periods_df['Date Range'].apply(lambda x: parse_date_range(x)[0])
periods_df['end_date'] = periods_df['Date Range'].apply(lambda x: parse_date_range(x)[1])

# Convert to datetime
periods_df['start_date'] = periods_df['start_date'].apply(
    lambda x: pd.to_datetime(convert_danish_to_english(x), format='%d. %B %Y', errors='coerce')
)
periods_df['end_date'] = periods_df['end_date'].apply(
    lambda x: pd.Timestamp.now() if x == 'nu' else pd.to_datetime(convert_danish_to_english(x), format='%d. %B %Y', errors='coerce')
)

# Function to find the period for a given date
def find_period(vote_date):
    for _, period in periods_df.iterrows():
        if period['start_date'] <= vote_date <= period['end_date']:
            return period['Period']
    return None

# Add period column
votings_df['Period'] = votings_df['møde_dato'].apply(find_period)


In [33]:
votings_df.to_csv('folketinget_votings_enrichedv2.csv', index=False)
print("\nData saved to 'folketinget_votings_enrichedv2.csv'")


Data saved to 'folketinget_votings_enrichedv2.csv'


In [32]:
votings_df.columns

Index(['afstemning_id', 'afstemning_nummer', 'afstemning_konklusion',
       'afstemning_vedtaget', 'afstemning_kommentar', 'afstemning_dato',
       'afstemning_opdateringsdato', 'afstemning_typeid', 'møde_id',
       'møde_dato', 'møde_titel', 'møde_nummer', 'møde_periodeid',
       'sagstrin_id', 'sagstrin_titel', 'sagstrin_typeid', 'sag_id', 'Period'],
      dtype='object')